# Two-mode van der Pol oscillator

This notebook follows the complete symbolic fpgen pipeline from a master equation to Hamiltonian and diffusion matrices, closed covariance dynamics, its Jacobian, and a scalar reduced equation. No numerical bifurcation solver is used.

In [ ]:
import sympy as sp
from IPython.display import Math, display
from fpgen import boson_modes, MasterEquation, LindbladChannel, derive_kramers_moyal

a, b = boson_modes("a", "b")
omega_a, omega_b, g = sp.symbols("omega_a omega_b g", real=True)
gamma_a, gamma_b, Gamma = sp.symbols("gamma_a gamma_b Gamma", real=True)
parameters = (omega_a, omega_b, g, gamma_a, gamma_b, Gamma)

def display_phase_matrix(matrix, variables):
    alpha, beta = sp.symbols(r"\alpha \beta", complex=True)
    replacements = {
        variables[0]: alpha, variables[1]: beta,
        variables[2]: sp.Symbol(r"\alpha^{*}", complex=True),
        variables[3]: sp.Symbol(r"\beta^{*}", complex=True),
    }
    return sp.Matrix(matrix).applyfunc(sp.simplify).xreplace(replacements)

def display_moment_expression(expression, dynamics):
    replacements = {
        item.symbol: sp.Symbol(item.latex_name, real=True)
        for item in dynamics.state_spec
    }
    if isinstance(expression, sp.MatrixBase):
        return sp.Matrix(expression).xreplace(replacements)
    return sp.sympify(expression).xreplace(replacements)

## 1. Master equation

The coherent part is

$$H=\omega_a a^\dagger a+\omega_b b^\dagger b+g(a^\dagger b+ab^\dagger).$$

Mode $a$ has linear gain $\gamma_a\mathcal L[a^\dagger]$ and nonlinear loss $\Gamma\mathcal L[a^2]$; mode $b$ has linear loss $\gamma_b\mathcal L[b]$.

In [ ]:
master = MasterEquation(
    modes=[a, b],
    hamiltonian=(
        omega_a*a.dag*a + omega_b*b.dag*b
        + g*(a.dag*b + a*b.dag)
    ),
    channels=[
        LindbladChannel(a.dag, gamma_a),
        LindbladChannel(a**2, Gamma),
        LindbladChannel(b, gamma_b),
    ],
)

## 2. Kramers--Moyal and Fokker--Planck equations

fpgen first performs the exact normal-to-flux rearrangement and only then truncates terms above second order. The nonlinear loss produces a third-order Wigner term, so the resulting Fokker--Planck equation is explicitly marked as truncated.

In [ ]:
km = derive_kramers_moyal(master, representation="wigner")
fpe = km.truncate(2)
langevin = fpe.to_langevin()
display(Math(
    r"\operatorname{ord}(\mathrm{KM})=" + str(km.order)
    + r",\qquad \operatorname{ord}(\mathrm{FPE})=2"
))

## 3. Hamiltonian and diffusion

The physical drift is $\dot{\boldsymbol\alpha}=-iH(\boldsymbol\alpha)\boldsymbol\alpha+\boldsymbol\xi$. Normal and anomalous correlations are read directly from the augmented FPE diffusion matrix.

In [ ]:
H = display_phase_matrix(langevin.physical_hamiltonian, langevin.variables)
D = display_phase_matrix(langevin.normal_correlation, langevin.variables)
M = display_phase_matrix(langevin.anomalous_correlation, langevin.variables)
display(Math(r"\mathbf H(\boldsymbol\alpha)=" + sp.latex(H)))
display(Math(r"\mathbf D(\boldsymbol\alpha)=" + sp.latex(D)))
display(Math(r"\mathbf M(\boldsymbol\alpha)=" + sp.latex(M)))

## 4. Closed covariance dynamics

For $R_{ij}=\langle\alpha_i\alpha_j^*\rangle$, the bilinear closure is exact at the retained moment level:

$$\dot R=-i(H(R)R-RH(R)^\dagger)+D(R).$$

The real state order is $(R_{aa},R_{bb},\operatorname{Re}R_{ab},\operatorname{Im}R_{ab})$.

In [ ]:
dynamics = langevin.to_covariance_dynamics(parameters=parameters)
R = display_moment_expression(dynamics.covariance, dynamics)
F = display_moment_expression(dynamics.rhs, dynamics)
J = display_moment_expression(dynamics.jacobian(), dynamics)
display(Math(r"\mathbf R=" + sp.latex(R)))
display(Math(r"\dot{\mathbf r}=\mathbf F(\mathbf r;\mathbf p)=" + sp.latex(F)))
display(Math(r"\mathbf J(\mathbf r;\mathbf p)=" + sp.latex(J)))

## 5. Choose the order parameter

Retain $q=R_{aa}$ and collect the other three coordinates into $\mathbf y$. Their stationary equations are affine in $\mathbf y$:

$$\mathbf h(q,\mathbf y;\mathbf p)=\mathbf A(q;\mathbf p)\mathbf y+\mathbf b(q;\mathbf p)=0.$$

In [ ]:
candidate = next(
    item for item in dynamics.find_linear_reductions(
        retained_dimension=1, equation_partitions="matching"
    )
    if item.retained_ids == ("r_diag_0",)
)
plan = dynamics.linear_reduce(candidate=candidate)
q = plan.retained_symbols[0]
y = sp.Matrix(plan.eliminated_symbols)
display(Math(r"q=R_{aa},\qquad \mathbf y=" + sp.latex(display_moment_expression(y, dynamics))))
display(Math(r"\mathbf A(q;\mathbf p)=" + sp.latex(display_moment_expression(plan.A, dynamics))))
display(Math(r"\mathbf b(q;\mathbf p)=" + sp.latex(display_moment_expression(plan.b, dynamics))))

## 6. Eliminate the remaining moments

On the regular branch $\operatorname{rank}A=3$, solve $\mathbf y_*=-A^{-1}b$ without explicitly constructing an inverse. Substitution into the retained equation gives the exact rational scalar dynamics $\dot q=G(q;\mathbf p)=P/Q$.

In [ ]:
reduced = plan.materialize(method="fraction_free")
y_star = display_moment_expression(reduced.eliminated_solution, dynamics)
G_reduced = display_moment_expression(reduced.reduced_residual[0], dynamics)
P = display_moment_expression(reduced.numerators[0], dynamics)
Q = display_moment_expression(reduced.denominators[0], dynamics)
display(Math(r"\mathbf y_*(q;\mathbf p)=" + sp.latex(y_star)))
display(Math(r"\dot q=G(q;\mathbf p)=" + sp.latex(G_reduced)))
display(Math(r"P(q;\mathbf p)=" + sp.latex(P)))
display(Math(r"Q(q;\mathbf p)=" + sp.latex(Q)))
display(Math(r"\deg_q P=" + str(sp.Poly(reduced.numerators[0], q).degree())))

## 7. High-order equilibrium conditions

On the regular branch and away from cleared denominators, an $m$-fold equilibrium satisfies $P=P'=\cdots=P^{(m-1)}=0$ with $P^{(m)}\ne0$. The VDP reduction is quartic, so it can support conditions through $m=4$ before parameters are specified.

In [ ]:
conditions = reduced.multiplicity_conditions(4)
display(Math(
    r"P=P'=P''=P'''=0,\qquad P^{(4)}\ne0"
))
display(Math(
    r"\det\mathbf A\ne0,\qquad Q\ne0"
))